# PANDA: Gene Regulatory Network Inference with netZooPy

This notebook demonstrates how to use the PANDA algorithm from the [netZooPy](https://netzoopy.readthedocs.io) package to infer a gene regulatory network by integrating gene expression, transcription factor binding motifs, and protein-protein interaction data.

**Reference:** Glass K, Huttenhower C, Quackenbush J, Yuan GC. *Passing Messages between Biological Networks to Refine Predicted Interactions.* PLoS ONE. 2013;8(5):e64832.

## 1. Installation

In [ ]:
# Install netZooPy (uncomment if not already installed)
# !pip install netZooPy

## 2. Import libraries

In [ ]:
from netZooPy.panda.panda import Panda
from netZooPy.lioness.lioness import Lioness
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 3. Define input data paths

We use the toy data bundled with netZooPy for demonstration. Replace these paths with your own data for real analyses.

**Input files:**
- `expression_file`: Tab-separated gene expression matrix (genes × samples)
- `motif_file`: TF–gene motif prior (TF, Gene, 0/1 weight)
- `ppi_file`: TF–TF protein-protein interactions

In [ ]:
# Locate the toy data shipped with netZooPy
import netZooPy
netzoo_path = os.path.dirname(netZooPy.__file__)
test_data = os.path.join(os.path.dirname(netzoo_path), 'tests', 'ToyData')

expression_file = os.path.join(test_data, 'ToyExpressionData.txt')
motif_file = os.path.join(test_data, 'ToyMotifData.txt')
ppi_file = os.path.join(test_data, 'ToyPPIData.txt')

print(f"Expression: {expression_file}")
print(f"Motif:      {motif_file}")
print(f"PPI:        {ppi_file}")

## 4. Inspect the input data

In [ ]:
# Expression data: rows = genes, columns = samples
expr_df = pd.read_csv(expression_file, sep='\t', index_col=0, header=None)
print(f"Expression matrix: {expr_df.shape[0]} genes × {expr_df.shape[1]} samples")
expr_df.head()

In [ ]:
# Motif prior: TF -> Gene -> Weight (0 or 1)
motif_df = pd.read_csv(motif_file, sep='\t', header=None, names=['TF', 'Gene', 'Weight'])
print(f"Motif edges: {len(motif_df)} (TFs: {motif_df['TF'].nunique()}, Genes: {motif_df['Gene'].nunique()})")
motif_df.head()

In [ ]:
# PPI data: TF1, TF2, interaction score
ppi_df = pd.read_csv(ppi_file, sep='\t', header=None, names=['TF1', 'TF2', 'Score'])
print(f"PPI edges: {len(ppi_df)}")
ppi_df.head()

## 5. Run PANDA

The `Panda` class handles data processing, co-expression computation, normalization, and the iterative message-passing algorithm in one call.

Key parameters:
- `remove_missing=False`: Keep all TFs/genes even if absent in one data source
- `save_memory=False`: Retain the full edge list (TF, Gene, Motif, Force) rather than just the adjacency matrix
- `keep_expression_matrix=True`: Needed if you plan to run LIONESS afterwards

In [ ]:
panda_obj = Panda(
    expression_file,
    motif_file,
    ppi_file,
    remove_missing=False,
    save_memory=False,
    keep_expression_matrix=True
)

## 6. Explore the PANDA network

The result is stored in `panda_obj.panda_network` — a DataFrame with columns:
- **TF**: transcription factor
- **Gene**: target gene
- **Motif**: prior edge weight (0/1)
- **Force**: PANDA edge weight (continuous)

In [ ]:
panda_network = panda_obj.panda_network
print(f"Network size: {len(panda_network)} edges")
print(f"TFs: {panda_network['tf'].nunique()}, Genes: {panda_network['gene'].nunique()}")
panda_network.head(10)

In [ ]:
# Distribution of PANDA edge weights
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(panda_network['force'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('PANDA Edge Weight (Force)')
ax.set_ylabel('Count')
ax.set_title('Distribution of PANDA Edge Weights')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Network analysis: Indegree and Outdegree

- **Indegree** (gene targeting): sum of edge weights into a gene — measures how strongly a gene is regulated overall
- **Outdegree** (TF targeting): sum of edge weights out of a TF — measures how broadly a TF regulates

In [ ]:
indegree = panda_obj.return_panda_indegree()
outdegree = panda_obj.return_panda_outdegree()

print("Top 10 genes by indegree (most regulated):")
print(indegree.sort_values('force', ascending=False).head(10))
print("\nTop 10 TFs by outdegree (broadest regulators):")
print(outdegree.sort_values('force', ascending=False).head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(indegree.sort_values('force', ascending=False).head(10)['gene'],
             indegree.sort_values('force', ascending=False).head(10)['force'])
axes[0].set_xlabel('Indegree (sum of incoming edge weights)')
axes[0].set_title('Top 10 Genes by Indegree')
axes[0].invert_yaxis()

axes[1].barh(outdegree.sort_values('force', ascending=False).head(10)['tf'],
             outdegree.sort_values('force', ascending=False).head(10)['force'])
axes[1].set_xlabel('Outdegree (sum of outgoing edge weights)')
axes[1].set_title('Top 10 TFs by Outdegree')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 8. Save results

In [ ]:
# Save the full PANDA network as a tab-separated file
output_file = 'panda_network.txt'
panda_obj.save_panda_results(output_file)
print(f"PANDA network saved to {output_file}")

## 9. Top network visualization

In [ ]:
# Plot the top 70 edges as a network graph
panda_obj.top_network_plot(top=70, file='panda_top_network.png')
print("Network plot saved to panda_top_network.png")

## 10. Single-sample networks with LIONESS

LIONESS (Linear Interpolation to Obtain Network Estimates for Single Samples) uses a leave-one-out approach on top of PANDA to estimate sample-specific GRNs. This allows differential network analysis between conditions or individuals.

In [ ]:
# Run LIONESS on top of the PANDA object
# Note: This computes one network per sample, which can be slow for large datasets.
# Here we run it on just the first 3 samples for demonstration.
lioness_obj = Lioness(
    panda_obj,
    start=1,
    end=3,
    save_dir='lioness_output',
    save_fmt='npy'
)

In [ ]:
# The LIONESS results are stored as an edge × sample matrix
print(f"LIONESS network shape: {lioness_obj.total_lioness_network.shape}")
print(f"  Rows = edges ({panda_network['tf'].nunique()} TFs × {panda_network['gene'].nunique()} genes)")
print(f"  Columns = samples")

In [ ]:
# Compare edge weight distributions between samples
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(lioness_obj.total_lioness_network.shape[1]):
    ax.hist(lioness_obj.total_lioness_network[:, i], bins=50, alpha=0.5, label=f'Sample {i+1}')
ax.set_xlabel('LIONESS Edge Weight')
ax.set_ylabel('Count')
ax.set_title('Sample-Specific Edge Weight Distributions')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Notes for real-world usage

- **Data sources**: Use the [GRAND database](https://grand.networkmedicine.org/) for pre-computed PANDA/LIONESS networks across GTEx tissues.
- **Motif data**: Motif scans can be obtained from databases like JASPAR or TRANSFAC, or use CisBP motif scans from the GRAND database.
- **PPI data**: STRING or BioGRID databases are common sources for TF–TF interactions.
- **Scaling**: For large datasets, use `computing='gpu'` if a CUDA-capable GPU is available, or `precision='single'` to reduce memory.
- **Gene naming**: Ensure consistent gene nomenclature across expression (columns) and motif (gene column), and consistent TF naming between motif (TF column) and PPI.
- **Processing mode**: Use `modeProcess='intersection'` to restrict to genes/TFs present in all data sources, or `'union'` (default) to include all and fill with zeros.

## 12. Summary

In this chapter we demonstrated:
1. How to set up input data for PANDA (expression, motif, PPI)
2. Running the PANDA message-passing algorithm via netZooPy
3. Interpreting the resulting GRN edge weights
4. Computing gene targeting (indegree) and TF targeting (outdegree) scores
5. Visualizing the inferred network
6. Extending to sample-specific networks with LIONESS

For more tutorials, see the [netZoo Netbooks](https://netbooks.networkmedicine.org/).